# Tuition ROI Analysis (Updated)

This notebook implements **Debt-Adjusted Net ROI**:

$$
\text{ROI}_{\text{Debt-Adjusted}} = \frac{\text{MidCareerPay} - (\text{MedianDebt} + \text{AnnualCost})}{\text{AnnualCost}}
$$

It expects the following files in the same folder or `data/raw/`:

- `Most-Recent-Cohorts-Institution_05192025.csv` (College Scorecard)
- `p24.xlsx` (income-by-education / PayScale-like workbook)

The notebook will:
- load and clean both files
- compute Debt-Adjusted Net ROI
- run regressions and compute confidence intervals
- create Seaborn visualizations
- save outputs to `./roi_outputs/`.


In [1]:
# Install required libraries if needed
# !pip install scikit-learn statsmodels seaborn openpyxl


In [2]:
# Imports and settings
import os, re, zipfile
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.api as sm
from sklearn.utils import resample
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [3]:
# Load the data files (with automatic local path detection)
search_dirs = [
    Path("."),
    Path("./data/raw"),
    Path("../data/raw"),
    Path(r"C:/Users/Donovan/Downloads"),
    Path(r"C:/higher-education-roi/data/raw")
]

csv_path = None
xls_path = None

for d in search_dirs:
    c = d / "Most-Recent-Cohorts-Institution_05192025.csv"
    x = d / "p24.xlsx"
    if c.exists() and csv_path is None:
        csv_path = str(c.resolve())
    if x.exists() and xls_path is None:
        xls_path = str(x.resolve())

if csv_path is None:
    for d in search_dirs:
        z = d / "Most-Recent-Cohorts-Institution_05192025.zip"
        if z.exists():
            print(f"Extracting CSV from {z}...")
            with zipfile.ZipFile(z, "r") as zip_ref:
                zip_ref.extractall(d)
            csv_path = str((d / "Most-Recent-Cohorts-Institution_05192025.csv").resolve())
            break

print('CSV path:', csv_path)
print('CSV exists?', os.path.exists(csv_path) if csv_path else False)
print('XLSX path:', xls_path)
print('XLSX exists?', os.path.exists(xls_path) if xls_path else False)

scorecard = pd.read_csv(csv_path, low_memory=False)
income_sheets = pd.read_excel(xls_path, sheet_name=None)
print('Scorecard rows,cols:', scorecard.shape)
print('Income workbook sheets:', list(income_sheets.keys()))


CSV path: C:\higher-education-roi\data\raw\Most-Recent-Cohorts-Institution_05192025.csv
CSV exists? True
XLSX path: C:\higher-education-roi\data\raw\p24.xlsx
XLSX exists? True


Scorecard rows,cols: (6429, 3306)
Income workbook sheets: ['p24']


In [4]:
# Heuristics to pick useful columns from the Scorecard
cols = scorecard.columns.tolist()
earn_candidates = [c for c in cols if re.search(r'earn.*p10', c, re.I)]
earn_col = 'MD_EARN_WNE_P10' if 'MD_EARN_WNE_P10' in scorecard.columns else (earn_candidates[0] if earn_candidates else None)
print('Selected earnings column:', earn_col)

cost_candidates = [c for c in cols if 'avg_net_price' in c.lower() or ('net' in c.lower() and 'price' in c.lower())]
if not cost_candidates:
    cost_candidates = [c for c in cols if 'tuition' in c.lower() and scorecard[c].dtype!='object']
cost_col = cost_candidates[0] if cost_candidates else None
print('Selected cost column (AnnualCost):', cost_col)

debt_candidates = [c for c in cols if 'grad_debt' in c.lower() or 'median_debt' in c.lower() or 'debt' in c.lower()]
debt_col = debt_candidates[0] if debt_candidates else None
print('Selected median debt column (best effort):', debt_col)

name_col = 'INSTNM' if 'INSTNM' in scorecard.columns else scorecard.columns[0]
print('Using name column:', name_col)


Selected earnings column: MD_EARN_WNE_P10
Selected cost column (AnnualCost): TUITIONFEE_IN
Selected median debt column (best effort): DEBT_MDN
Using name column: INSTNM


In [5]:
# Extract a reasonable HS baseline from the income workbook (sheet 'p24' or first sheet)
p24 = None
if 'p24' in income_sheets:
    p24 = income_sheets['p24']
else:
    first_sheet = list(income_sheets.keys())[0]
    p24 = income_sheets[first_sheet]

nums = pd.to_numeric(p24.stack().astype(str).str.replace(r'[^0-9.]','', regex=True), errors='coerce').dropna()
nums = nums[(nums>15000) & (nums<60000)]
hs_baseline = float(nums.median()) if len(nums)>0 else 35000.0
print('Estimated HS baseline used for earn_premium:', hs_baseline)


Estimated HS baseline used for earn_premium: 38420.0


In [6]:
# Build ROI dataframe: Debt-Adjusted Net ROI = (MidCareerPay - (MedianDebt + AnnualCost)) / AnnualCost
roi = pd.DataFrame()
roi['school'] = scorecard[name_col].astype(str)

if earn_col and earn_col in scorecard.columns:
    roi['midcareer_pay'] = pd.to_numeric(scorecard[earn_col], errors='coerce')
else:
    any_earn = [c for c in scorecard.columns if 'earn' in c.lower()]
    roi['midcareer_pay'] = pd.to_numeric(scorecard[any_earn[0]], errors='coerce') if any_earn else np.nan

if cost_col and cost_col in scorecard.columns:
    roi['annual_cost'] = pd.to_numeric(scorecard[cost_col], errors='coerce')
else:
    roi['annual_cost'] = np.nan

if debt_col and debt_col in scorecard.columns:
    roi['median_debt'] = pd.to_numeric(scorecard[debt_col], errors='coerce')
else:
    debt_try = None
    for c in ['GRAD_DEBT_MDN','GRAD_DEBT_MDN_SUPP','md_debt']:
        if c in scorecard.columns:
            debt_try = c
            break
    roi['median_debt'] = pd.to_numeric(scorecard[debt_try], errors='coerce') if debt_try else np.nan

roi['total_cost'] = roi['annual_cost']
roi['roi_debt_adjusted'] = (roi['midcareer_pay'] - (roi['median_debt'] + roi['annual_cost'])) / roi['annual_cost']
roi['earn_premium'] = roi['midcareer_pay'] - hs_baseline
roi['break_even_years'] = np.where(roi['earn_premium']>0, (roi['median_debt']+roi['annual_cost'])/roi['earn_premium'], np.nan)

if 'CONTROL' in scorecard.columns:
    roi['sector'] = scorecard['CONTROL'].map({1:'Public',2:'Private_nonprofit',3:'Private_forprofit'})
else:
    roi['sector'] = 'Unknown'

roi = roi.replace([np.inf, -np.inf], np.nan)
print('ROI dataframe built. Sample:')
display(roi.head(10))
display(roi[['midcareer_pay','annual_cost','median_debt','roi_debt_adjusted','earn_premium','break_even_years']].describe().transpose())


ROI dataframe built. Sample:


,school,midcareer_pay,annual_cost,median_debt,total_cost,roi_debt_adjusted,earn_premium,break_even_years,sector
0,Alabama A & M University,40628.0,10024.0,16600.0,10024.0,1.397047,2208.0,12.057971,Public
1,University of Alabama at Birmingham,54501.0,8832.0,15832.0,8832.0,3.378284,16081.0,1.533735,Public
2,Amridge University,37621.0,NaN,13385.0,NaN,NaN,-799.0,NaN,Private_nonprofit
3,University of Alabama in Huntsville,61767.0,11770.0,13905.0,11770.0,3.066440,23347.0,1.099713,Public
4,Alabama State University,34502.0,11248.0,17500.0,11248.0,0.511558,-3918.0,NaN,Public
5,The University of Alabama,59221.0,11900.0,17986.0,11900.0,2.465126,20801.0,1.436758,Public
6,Central Alabama Community College,33506.0,5040.0,5500.0,5040.0,4.556746,-4914.0,NaN,Public
7,Athens State University,50273.0,NaN,14861.0,NaN,NaN,11853.0,NaN,Public
8,Auburn University at Montgomery,44391.0,9436.0,13119.0,9436.0,2.314116,5971.0,3.777424,Public
9,Auburn University,65337.0,12536.0,17750.0,12536.0,2.796027,26917.0,1.125163,Public


,count,mean,std,min,25%,50%,75%,max
midcareer_pay,5280.0,43508.301136,17033.197929,8579.000000,31830.000000,40567.500000,51994.000000,143372.000000
annual_cost,3729.0,17237.652722,15644.112456,600.000000,5688.000000,11790.000000,23186.000000,69330.000000
median_debt,5283.0,11268.762446,5190.790837,1932.000000,7000.000000,9500.000000,15000.000000,38980.000000
roi_debt_adjusted,3278.0,3.620996,5.768679,-1.440886,0.200020,1.752985,4.994277,88.005291
earn_premium,5280.0,5088.301136,17033.197929,-29841.000000,-6590.000000,2147.500000,13574.000000,104952.000000
break_even_years,2495.0,9.050063,86.339064,0.142652,1.521019,2.736378,4.769257,3966.285714


In [7]:
# Bootstrap 95% CI for mean earn_premium across institutions
vals = roi['earn_premium'].dropna().values
if len(vals)>0:
    rng = np.random.default_rng(42)
    nboots = 2000
    boot_means = [np.mean(rng.choice(vals, size=len(vals), replace=True)) for _ in range(nboots)]
    ci_low = np.percentile(boot_means, 2.5)
    ci_high = np.percentile(boot_means, 97.5)
    print(f'Mean earn_premium = {np.mean(vals):.2f}, bootstrap 95% CI = [{ci_low:.2f}, {ci_high:.2f}]')
else:
    print('Not enough earn_premium values for bootstrap.')


Mean earn_premium = 5088.30, bootstrap 95% CI = [4651.49, 5549.29]


In [8]:
# Regression: predict Debt-Adjusted Net ROI using annual_cost, median_debt, midcareer_pay (and sector)
reg_df = roi.dropna(subset=['roi_debt_adjusted','annual_cost','midcareer_pay']).copy()
reg_df = pd.get_dummies(reg_df, columns=['sector'], drop_first=True, dtype=float)
X_cols = ['annual_cost','midcareer_pay','median_debt'] + [c for c in reg_df.columns if c.startswith('sector_')]
for c in X_cols:
    if c not in reg_df.columns:
        reg_df[c] = 0.0
    else:
        reg_df[c] = pd.to_numeric(reg_df[c], errors='coerce').fillna(0.0).astype(float)

X = reg_df[X_cols]
X = sm.add_constant(X)
y = pd.to_numeric(reg_df['roi_debt_adjusted'], errors='coerce').astype(float)

if len(reg_df) > 30:
    model = sm.OLS(y, X).fit(cov_type='HC3')
    print(model.summary())
else:
    model = None
    print('Too few rows for ROI regression.')

# Log transform variant (log1p of ROI where ROI > -0.9)
reg_log = reg_df[reg_df['roi_debt_adjusted'] > -0.9].copy()
if len(reg_log) > 30:
    reg_log['log_roi_plus1'] = np.log1p(reg_log['roi_debt_adjusted'])
    X2 = reg_log[X_cols]
    X2 = sm.add_constant(X2)
    y_log = pd.to_numeric(reg_log['log_roi_plus1'], errors='coerce').astype(float)
    model_log = sm.OLS(y_log, X2).fit(cov_type='HC3')
    print('\nLog-ROI model summary:')
    print(model_log.summary())
else:
    model_log = None
    print('Too few rows for log ROI regression.')


                            OLS Regression Results                            
Dep. Variable:      roi_debt_adjusted   R-squared:                       0.415
Model:                            OLS   Adj. R-squared:                  0.414
Method:                 Least Squares   F-statistic:                     326.1
Date:                Fri, 28 Aug 2026   Prob (F-statistic):          4.91e-284
Time:                        16:01:57   Log-Likelihood:                -9516.6
No. Observations:                3278   AIC:                         1.905e+04
Df Residuals:                    3272   BIC:                         1.908e+04
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [9]:
# Create output plots & CSVs (Seaborn styling)
outdir = os.path.join(os.getcwd(), 'roi_outputs')
os.makedirs(outdir, exist_ok=True)

# Histogram of ROI
plt.figure(figsize=(8,5))
sns.histplot(roi['roi_debt_adjusted'].dropna(), bins=80, kde=False)
plt.title('Distribution of Debt-Adjusted Net ROI')
plt.xlabel('Debt-Adjusted Net ROI (Multiple)')
plt.savefig(os.path.join(outdir,'roi_hist.png'), bbox_inches='tight')
plt.close()

# Top 15 ROI bar plot
top15 = roi.replace([np.inf,-np.inf],np.nan).dropna(subset=['roi_debt_adjusted']).nlargest(15,'roi_debt_adjusted')
plt.figure(figsize=(10,6))
sns.barplot(x='roi_debt_adjusted', y='school', data=top15, hue='school', palette='viridis', legend=False)
plt.title('Top 15 schools by Debt-Adjusted Net ROI')
plt.xlabel('Debt-Adjusted Net ROI')
plt.ylabel('School')
plt.tight_layout()
plt.savefig(os.path.join(outdir,'top15_roi.png'))
plt.close()

# Scatter: annual cost vs ROI colored by sector
plt.figure(figsize=(8,6))
sns.scatterplot(data=roi, x='annual_cost', y='roi_debt_adjusted', hue='sector', alpha=0.7, s=40)
plt.title('Annual cost vs Debt-Adjusted Net ROI (colored by sector)')
plt.xlabel('Annual cost')
plt.ylabel('Debt-Adjusted Net ROI')
plt.savefig(os.path.join(outdir,'cost_vs_roi_scatter.png'))
plt.close()

# Save CSVs
top15[['school','annual_cost','median_debt','midcareer_pay','roi_debt_adjusted']].to_csv(os.path.join(outdir,'top15_roi.csv'), index=False)
roi.to_csv(os.path.join(outdir,'roi_full.csv'), index=False)
print('Plots and CSVs saved to', outdir)


Plots and CSVs saved to C:\higher-education-roi\notebooks\roi_outputs


In [10]:
# Save regression outputs if present
if model is not None:
    with open(os.path.join(outdir, 'roi_ols_summary.txt'), 'w', encoding='utf-8') as f:
        f.write(model.summary().as_text())
if model_log is not None:
    with open(os.path.join(outdir, 'roi_logols_summary.txt'), 'w', encoding='utf-8') as f:
        f.write(model_log.summary().as_text())

print(f'Notebook run complete. Check {outdir} for saved artifacts.')


Notebook run complete. Check C:\higher-education-roi\notebooks\roi_outputs for saved artifacts.
